# WARNING: Still need to check the descriptions here, some are still incorrect. Code should be fine

# Dataset Statistics: QC Metrics

Evaluates three complementary quality dimensions that characterise the reliability and spatial consistency of measurements across all participating laboratories:

1. **Missingness / completeness** — how completely each feature is detected across samples, and how completely each sample covers the feature space, at both single-cell and QC injection level.
2. **Plate heatmap** — spatial distribution of identification counts across each physical 16 × 24 well plate, overlaying single-cell and QC injections to expose within-plate gradients or positional effects.
3. **Retention time (RT) stability** — the shape of the elution profile (proportion of identifications per RT bin) across samples, used to detect gradient failures, carry-over, or systematic shifts between labs.

These metrics are used in **Figure 4** of the manuscript. The completeness values computed here are also stored back into the AnnData objects and used downstream as a sample-level QC filter.

**Inputs**

| File | Description |
|---|---|
| `Data/AnnData/adata_precursor.h5ad` | Precursor-level quantification, single cells |
| `Data/AnnData/adata_protein.h5ad` | Protein-level quantification, single cells |
| `Data/AnnData/adata_precursor_qc.h5ad` | Precursor-level quantification, QC injections |
| `Data/AnnData/adata_protein_qc.h5ad` | Protein-level quantification, QC injections |
| `Data/Parquet/search_results.parquet` | Long-format search results (needed for per-precursor RT values) |

**Outputs** (`Figures/Figure_4/`)

| Panel | File | Description |
|---|---|---|
| A | `RT_Stability/RT_stability_overview_SC.png` | RT stability overview, single-cell conditions, coloured by lab |
| B | `RT_Stability/RT_stability_overview_QC.png` | RT stability overview, QC conditions, coloured by lab |
| C | `Plate_Heatmap/plate_heatmap_protein_{lab}.png` | Per-lab protein ID plate heatmap |
| D | `Plate_Heatmap/plate_heatmap_precursor_{lab}.png` | Per-lab precursor ID plate heatmap |
| E | `Completeness/completeness_precursors_SC.png` | Precursor completeness curves, single cells |
| E | `Completeness/completeness_precursors_QC.png` | Precursor completeness curves, QC injections |
| F | `Completeness/completeness_proteins_SC.png` | Protein completeness curves, single cells |
| F | `Completeness/completeness_proteins_QC.png` | Protein completeness curves, QC injections |
| Supp. | `RT_Stability/RT_stability_{lab}.png` | Per-lab RT stability coloured by condition |

**AnnData updates** — the following columns are written back before saving:

| Object | Slot | Column | Description |
|---|---|---|---|
| precursor / protein | `obs` | `completeness_sample_SC` | Fraction of lab-detected features found in each single cell |
| precursor / protein | `obs` | `completeness_sample_QC` | Fraction of lab-detected features found in each QC injection |
| precursor / protein | `var` | `completeness_feature_{lab}_SC/QC` | Per-lab detection rate across samples for each feature |
| precursor / protein | `obs` | `completeness_outlier` | `True` if sample fails both SC and QC completeness thresholds |

---

## Imports

In [27]:
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
import anndata

tqdm.pandas()

## 0. Path Configuration

Input AnnData and Parquet files are read from `ROOT_OUT/AnnData/` and `ROOT_OUT/Parquet/`. Figures are written to `FIGURES_OUT/Figure_4/` with one sub-directory per panel (`RT_Stability/`, `Plate_Heatmap/`, `Completeness/`). Both root directories can be overridden via environment variables (`SCI_DATA_DIR`, `SCI_FIGURES_DIR`) for running the notebook on different machines or in automated pipelines.

In [28]:
NOTEBOOK_DIR = Path().resolve()
ROOT_OUT      = NOTEBOOK_DIR / "../../Data"
ROOT_OUT      = ROOT_OUT.resolve()
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Allow user override via environment variable
ROOT_OUT = Path(os.environ.get("SCI_DATA_DIR", ROOT_OUT))

ROOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {ROOT_OUT}")

FIGURES_OUT = NOTEBOOK_DIR / "../../Figures"
FIGURES_OUT = FIGURES_OUT.resolve()
FIGURES_OUT.mkdir(parents=True, exist_ok=True)

# Overwrite with environment variable
FIGURES_OUT = Path(os.environ.get("SCI_FIGURES_DIR", FIGURES_OUT))
print(f"Figures directory: {FIGURES_OUT}")

ANNDATA_IN = os.path.join(
    ROOT_OUT,
    'AnnData'
)
PARQUET_IN = os.path.join(
    ROOT_OUT,
    'Parquet'
)

Output directory: /home/sam/SCI/SCI/Data
Figures directory: /home/sam/SCI/SCI/Figures


## 1. Load Data

Four AnnData objects are loaded: single-cell data at precursor level (3,114 samples × 72,061 precursors) and protein level (3,114 × 7,231), plus the equivalent QC injection objects. The long-format `search_results.parquet` is also loaded because RT values (`EG.ApexRT`) are stored there at the individual precursor-identification level, not in the pivoted AnnData matrix.

Identification counts (`n_precursors`, `n_proteins`) are computed for the QC objects here so they are available for the plate heatmap visualisation.

In [29]:
adata_precursor = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor.h5ad')
)
adata_protein = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_protein.h5ad')
)

adata_precursor_qc = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor_qc.h5ad')
)
adata_protein_qc = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_protein_qc.h5ad')
)

report = pd.read_parquet(
    os.path.join(PARQUET_IN, 'search_results.parquet')
)

In [30]:
# Add counts
adata_precursor_qc.obs['n_precursors'] = np.sum(~np.isnan(
    adata_precursor_qc[:, ~adata_precursor_qc.var['Contaminant']].X
), axis=1)
adata_protein_qc.obs['n_proteins'] = np.sum(~np.isnan(
    adata_protein_qc[:, ~adata_protein_qc.var['Contaminant']].X
), axis=1)

In [31]:
adata_precursor

AnnData object with n_obs × n_vars = 3114 × 72061
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type', 'n_precursors', 'completeness_sample_SC', 'completeness_sample_QC', 'completeness_outlier', 'n_precursors_withCont', 'pct_contaminant'
    var: 'PG.ProteinGroups', 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.

In [32]:
adata_protein

AnnData object with n_obs × n_vars = 3114 × 7231
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type', 'n_proteins', 'completeness_sample_SC', 'completeness_sample_QC', 'completeness_outlier', 'n_proteins_withCont', 'pct_contaminant'
    var: 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'FG.PrecursorIon', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.iRTPre

## 2. QC Tiers

Quality assessment is structured in three tiers that together span the spatial, quantitative, and chromatographic dimensions of data quality. Each tier is computed separately for **single-cell conditions** (K562, HEK293T) and **QC injection conditions** (QC - HeLa, site-specific QC), allowing cross-context comparison of the same metric.

### 2.1. Missingness Pattern — Completeness Curves

Completeness curves show the trade-off between stringency and data retention when applying a missing-value filter. Two complementary perspectives are computed per lab:

- **Feature-level completeness** (left panel) — for each threshold *t* (0–100%), the fraction of features detected in ≥ *t*% of samples. A feature with 50% completeness is quantified in half the cells. The curve's shape reveals whether missingness is diffuse (gradual decline) or concentrated in a few stochastically undetected features (steep drop near 0%).
- **Sample-level completeness** (right panel) — for each threshold *t*, the fraction of samples that have ≥ *t*% of the lab's detectable features. A sample at 25% completeness has quantified one quarter of what that lab typically detects. This identifies near-empty injections (failed electrospray, clogged column, dispensing error).

Each lab is shown as a coloured line; the dashed black line represents the combined curve across all labs. The 50% reference line provides a natural anchor for comparison.

All four objects (precursor + protein × SC + QC conditions) are processed, and the resulting per-sample and per-feature completeness values are stored back into `adata.obs` and `adata.var` via `store_completeness`, with suffixes `_SC` and `_QC` to keep the two condition sets distinguishable.

In [33]:
from SCI_py.metrics.completeness import compute_completeness, compute_combined_curve, store_completeness
from SCI_py.plot.completeness import plot_completeness

#### Single-cell conditions (K562, HEK293T)

Completeness is evaluated only within the two primary cell-line conditions. Per-sample completeness is stored as `completeness_sample_SC` in `adata.obs`; per-feature detection rates per lab are stored as `completeness_feature_{lab}_SC` in `adata.var`.

In [34]:
# Precursor
conditions = ['K562', 'HEK293T']
thresholds, retention, samples = compute_completeness(adata_precursor, conditions)
combined_feat = compute_combined_curve(adata_precursor, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_precursor, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds,
    retention,
    samples,
    combined_feat,
    combined_samp,
    label='Precursors',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_precursors_SC.png',
    show=False
)
store_completeness(
    adata=adata_precursor,
    conditions=conditions,
    conditions_suffix='SC'
)

# Protein
thresholds, retention, samples = compute_completeness(adata_protein, conditions)
combined_feat = compute_combined_curve(adata_protein, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_protein, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds,
    retention,
    samples,
    combined_feat,
    combined_samp,
    label='Proteins',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_proteins_SC.png',
    show=False
)
store_completeness(
    adata=adata_protein,
    conditions=conditions,
    conditions_suffix='SC'
)

#### QC injection conditions (QC - HeLa, site-specific QC)

The same completeness analysis is repeated for the HeLa and site-specific QC injections. Because QC injections use bulk material (not single cells), completeness should be substantially higher — any QC injection with low completeness is a stronger signal of an instrument or LC problem than the same completeness in a single cell. Results stored with suffix `_QC`.

In [35]:
# Precursor
conditions = ['QC - Hela', 'Site specific QC']
thresholds, retention, samples = compute_completeness(adata_precursor, conditions)
combined_feat = compute_combined_curve(adata_precursor, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_precursor, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds,
    retention,
    samples,
    combined_feat,
    combined_samp,
    label='Precursors',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_precursors_QC.png',
    show=False
)
store_completeness(
    adata=adata_precursor,
    conditions=conditions,
    conditions_suffix='QC'
)

# Protein
thresholds, retention, samples = compute_completeness(adata_protein, conditions)
combined_feat = compute_combined_curve(adata_protein, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_protein, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds,
    retention,
    samples,
    combined_feat,
    combined_samp,
    label='Proteins',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_proteins_QC.png',
    show=False
)
store_completeness(
    adata=adata_protein,
    conditions=conditions,
    conditions_suffix='QC'
)

### 2.2. Plate Heatmap

Visualises per-well identification counts spatially on the physical plate layout. The plate is drawn as a 16 (rows) × 24 (columns) grid; each well is rendered as a coloured rectangle with the identification count printed inside.

**Colour encoding** — wells are filled with the condition colour from `CONDITION_COLORS`. The colour alpha (opacity) is linearly scaled from 0.3 (minimum count) to 1.0 (maximum count within the plate), so the intensity of the colour directly encodes how many features were identified. Empty or missing wells are shown at α = 0.1.

**Section boundaries** — white grid lines separate the distinct well sections of the SCI plate design (e.g. cell wells, blank wells, QC wells), making it easy to verify that identification depth tracks expected plate layout patterns.

**QC overlay** — the `extra_adata` argument injects the QC injection observations into the same plate visualisation. QC wells that were run separately are merged with the single-cell plate on (X, Y) position, so the full physical plate is shown in a single figure. This is particularly useful for diagnosing whether low-count regions are confined to cell wells (dispensing issue) or span QC wells too (LC/instrument issue).

One figure is produced per lab, at both protein and precursor level.

In [36]:
from SCI_py.plot.plate_heatmap import plot_plate_heatmap

In [37]:
plot_plate_heatmap(
    adata_protein,
    count_col   = 'n_proteins',
    extra_adata = adata_protein_qc,
    label       = 'Proteins',
    save_dir    = os.path.join(FIGURES_OUT, 'Figure_4', 'Plate_Heatmap'),
    filename_fmt = 'plate_heatmap_protein_{lab}.png',
    show=False
)

plot_plate_heatmap(
    adata_precursor,
    count_col   = 'n_precursors',
    extra_adata = adata_precursor_qc,
    label       = 'Precursors',
    save_dir    = os.path.join(FIGURES_OUT, 'Figure_4', 'Plate_Heatmap'),
    filename_fmt = 'plate_heatmap_precursor_{lab}.png',
    show=False
)

### 2.3. Retention Time Stability

Chromatographic stability is assessed by computing the proportion of identified precursors falling into each RT bin for every sample, then comparing the distribution shapes across samples and labs. A stable chromatographic run produces an elution profile of consistent shape; deviations indicate gradient drift, injection failures, or carry-over. Here, we do include the contaminant proteins as we are solely interested in the identifications across the elution profile, irrespective of the source of these proteins.

**Method** — for each sample, the `EG.ApexRT` values from the long-format report are normalised to [0, 1] within that sample (`normalize_rt=True`), binned into 25 equal-width bins, and the fraction of identifications per bin is computed. Normalising per sample removes the effect of absolute gradient length differences between labs, making elution profiles directly comparable.

**Outlier exclusion** — samples flagged as `completeness_outlier` (< 25% completeness in the SC and QC conditions; see threshold cell below) are excluded from RT analysis. Near-empty injections produce noisy, uninformative RT profiles and would otherwise dominate the visual distribution.

**Two figure types** are produced:
- **Per-lab supplementary figures** (coloured by condition) — individual sample curves (α = 0.2) overlaid with per-condition group mean curves (linewidth = 2.5). These reveal whether RT instability is condition-specific or lab-wide.
- **Overview figures** (coloured by lab) — group mean curves only, with individual curves suppressed (`show_individual=False`). Generated separately for single-cell (K562, HEK293T) and QC (HeLa, site-specific QC) conditions to allow direct cross-lab comparison without per-sample noise.

In [38]:
from SCI_py.metrics.rt_stability import compute_rt_proportions
from SCI_py.plot.rt_stability import plot_rt_stability

from SCI_py.constants.maps import LABS

#### Completeness outlier threshold

Samples are flagged as `completeness_outlier = True` if the samples don't pass the set thresholds for their respective set (e.g. SC or QC samples). The threshold of **25%** is selected from the completeness curves above — it sits below the main distribution of well-measured samples and above the cluster of clearly failed injections.

The outlier flag is propagated from the precursor to the protein AnnData by index alignment, ensuring consistent sample exclusion at both quantification levels.

In [39]:
adata_precursor.obs['completeness_outlier'] = ~(
    (adata_precursor.obs['completeness_sample_SC'] > 0.25) |
    (adata_precursor.obs['completeness_sample_QC'] > 0.25)
)

adata_protein.obs['completeness_outlier'] = (
    adata_precursor.obs['completeness_outlier']
    .reindex(adata_protein.obs_names)
)

In [40]:
rt_props = []

conditions = ['HEK293T', 'K562', 'QC - Hela', 'Site specific QC']

for lab_name in LABS:
    rt_props_lab = compute_rt_proportions(
        report=report,
        sample_ids=adata_precursor.obs[
            (adata_precursor.obs['plate_idx']==lab_name) &
            (~adata_precursor.obs['completeness_outlier'])
        ].index.tolist(),
        rt_col='EG.ApexRT',
        conditions=conditions,
        bins=25,
        normalize_rt=True
    )
    rt_props_lab = rt_props_lab.merge(adata_precursor.obs.reset_index(), how='left')
    rt_props.append(rt_props_lab)
    
    # Plot the retention time stability
    plot_rt_stability(
        rt_props=rt_props_lab,
        color_by='source name',
        title=lab_name,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
        filename=f"RT_stability_{lab_name}.png",
        show=False
    )

rt_props = pd.concat(rt_props, ignore_index=True)

In [41]:
plot_rt_stability(
    rt_props=rt_props[rt_props['source name'].isin(['K562', 'HEK293T'])],
    color_by='plate_idx',
    title=f'overview_SC',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
    filename=f"RT_stability_overview_SC.png",
    show_individual=False,
    show=False
)
plot_rt_stability(
    rt_props=rt_props[rt_props['source name'].isin(['Site specific QC', 'QC - Hela'])],
    color_by='plate_idx',
    title=f'overview_QC',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
    filename=f"RT_stability_overview_QC.png",
    show_individual=False,
    show=False
)

## 3. Identity of Proteins in Control Conditions

Control wells on each SCI plate serve distinct purposes: they separate biological signal from sample preparation artefacts, instrument background, and cross-contamination. The helper `get_proteins_by_condition` queries each control type and, for each lab, returns the protein accessions detected in at least one well of that condition. The result is a per-lab DataFrame of detected proteins, from which the most frequently recurring accessions are extracted.

**Why this matters** — protein identifications in negative-control wells constrain the interpretable proteome: any protein consistently appearing in blanks may reflect carry-over, ambient contamination, or reagent impurities rather than genuine cellular signal. The per-lab breakdown also flags whether contamination is instrument-specific (suggesting a fluidic or column issue) or universal across labs (suggesting a reagent or protocol artefact).

The four control types analysed below differ in what they hold and where they appear in the experimental workflow:

| Control | Content | Purpose |
|---|---|---|
| CellenONE blank | Empty well, no cell dispensed | Detects dispensing carry-over and reagent background |
| Site-specific blank | Lab-defined blank, may contain buffer | Captures lab-specific preparation background |
| Processing control | Bulk lysate processed alongside single cells | Monitors lysis and digestion efficiency |
| Sorting control | Cells sorted without downstream processing | Measures cell handling and sort contamination |

In [66]:
def get_proteins_by_condition(
    adata,
    source_name: list,
):
    adata_selection = adata[
        adata.obs['source name'].isin(source_name),
        ~adata.var['Contaminant']
    ]

    # For each lab, get the protein accessions detected in at least one blank cell
    protein_list = {}
    for lab in LABS:
        lab_idx = adata_selection.obs[adata_selection.obs['plate_idx'] == lab].index
        if len(lab_idx) == 0:
            continue

        X_lab        = adata_selection[lab_idx, :].X
        detected     = np.sum(~np.isnan(X_lab), axis=0) > 0
        protein_ids  = adata_selection.var.loc[detected, 'PG.ProteinAccessions'].tolist()
        protein_list[lab] = protein_ids
        print(f"{lab}: {len(protein_ids)} proteins detected in blanks")

    # Show as a DataFrame for easy inspection
    max_len = max(len(v) for v in protein_list.values())
    protein_df = pd.DataFrame({
        lab: pd.Series(proteins)
        for lab, proteins in protein_list.items()
    })
    return protein_df

### 3.1. CellenONE Blank

Wells where no cell was dispensed by the CellenONE instrument. Any protein identified here represents reagent background, spray carry-over from neighbouring wells, or signal from ambient aerosol. The frequency distribution shows how many labs detect the same protein (a protein found by all labs is a likely ubiquitous contaminant; one found by a single lab suggests a lab-specific source).

In [73]:
cellenone_blank = get_proteins_by_condition(
    adata_protein,
    source_name=['Cellenone blank']
)

print(cellenone_blank.melt().dropna().value_counts('value').value_counts())

print('\nMost occuring proteins: ')
print(cellenone_blank.melt().dropna().value_counts('value').head(15).index.tolist())

Astral_1: 1 proteins detected in blanks
Astral_2: 209 proteins detected in blanks
Astral_3: 12 proteins detected in blanks
Astral_4: 38 proteins detected in blanks
Astral_5: 8 proteins detected in blanks
Astral_6: 28 proteins detected in blanks
timsTOF_1: 15 proteins detected in blanks
timsTOF_2: 18 proteins detected in blanks
timsTOF_3: 29 proteins detected in blanks
count
1    319
2     14
4      2
3      1
Name: count, dtype: int64

Most occuring proteins: 
['Q562R1', 'Q15828', 'P06899', 'P12236', 'Q8TF72', 'O00481', 'P63261', 'Q8NCU4', 'P35030', 'P61626', 'O14556', 'O75952', 'P39687', 'P62807', 'Q96L96']


### 3.2. Site-specific Blank

Blank wells defined by each participating lab, typically containing sample preparation buffer without biological material. Proteins detected here capture lab-specific background — for example, proteins from the hands of a researcher, common laboratory reagents, or environmental keratin sources that differ by site.

In [74]:
site_specific_blanks = get_proteins_by_condition(
    adata_protein,
    source_name=['Site specific blanks']
)

print(site_specific_blanks.melt().dropna().value_counts('value').value_counts())

print('\nMost occuring proteins: ')
print(site_specific_blanks.melt().dropna().value_counts('value').head(15).index.tolist())

Astral_1: 7 proteins detected in blanks
Astral_2: 797 proteins detected in blanks
Astral_3: 17 proteins detected in blanks
Astral_4: 41 proteins detected in blanks
Astral_5: 1340 proteins detected in blanks
Astral_6: 29 proteins detected in blanks
timsTOF_1: 7 proteins detected in blanks
timsTOF_2: 22 proteins detected in blanks
timsTOF_3: 28 proteins detected in blanks
count
1    966
2    634
3     18
Name: count, dtype: int64

Most occuring proteins: 
['Q562R1', 'Q15843', 'Q16718', 'P62979', 'P62826', 'P62857', 'P62318', 'P06899', 'P23528', 'P63261', 'Q9HCM7', 'P09104', 'P0C0S5;Q71UI9', 'Q8TF72', 'P35030']


### 3.3. Processing Control

A bulk cell lysate (same cell line as the single-cell experiments) processed in parallel through the complete sample preparation workflow on the same plate. Because this control contains real protein material at high abundance, detected proteins here are expected and informative: a rich, reproducible profile confirms the lysis and digestion steps are functioning. Any protein absent from the processing control but present in single cells would be a strong candidate for a dispensing or protocol artefact.

In [75]:
processing_control = get_proteins_by_condition(
    adata_protein,
    source_name=['Processing control']
)

print(processing_control.melt().dropna().value_counts('value').value_counts())

print('\nMost occuring proteins: ')
print(processing_control.melt().dropna().value_counts('value').head(15).index.tolist())

Astral_1: 73 proteins detected in blanks
Astral_2: 266 proteins detected in blanks
Astral_3: 377 proteins detected in blanks
Astral_4: 680 proteins detected in blanks
Astral_5: 35 proteins detected in blanks
Astral_6: 1368 proteins detected in blanks
timsTOF_1: 280 proteins detected in blanks
timsTOF_2: 565 proteins detected in blanks
timsTOF_3: 2664 proteins detected in blanks
count
1    2122
2     788
3     238
4     113
5      79
6      61
7      52
8      32
9       7
Name: count, dtype: int64

Most occuring proteins: 
['P01040', 'P25311', 'P07477', 'P31944', 'P61626', 'P35030', 'P05109', 'P22735', 'P47929', 'Q5T749', 'Q15828', 'P01857', 'P04083', 'P62979', 'Q08188']


### 3.4. Sorting Control

Cells that were sorted by the fluorescence-activated cell sorter (FACS) but not subjected to downstream processing (no lysis, no digestion). Proteins detected in sorting controls reflect biological material carried over from the sort buffer, fluorescent dye contaminants, or signal from ruptured cells during sorting. Comparing sorting controls to CellenONE blanks isolates the contribution of the sorting step to the overall background.

In [76]:
sorting_control = get_proteins_by_condition(
    adata_protein,
    source_name=['Sorting control']
)

print(sorting_control.melt().dropna().value_counts('value').value_counts())

print('\nMost occuring proteins: ')
print(sorting_control.melt().dropna().value_counts('value').head(15).index.tolist())

Astral_1: 2319 proteins detected in blanks
Astral_2: 3720 proteins detected in blanks
Astral_3: 417 proteins detected in blanks
Astral_4: 880 proteins detected in blanks
Astral_5: 994 proteins detected in blanks
Astral_6: 1169 proteins detected in blanks
timsTOF_1: 80 proteins detected in blanks
timsTOF_2: 673 proteins detected in blanks
timsTOF_3: 1553 proteins detected in blanks
count
1    1877
2     960
3     612
4     420
5     301
6     170
8     130
7     117
9      12
Name: count, dtype: int64

Most occuring proteins: 
['P81605', 'P01040', 'P07339', 'P07477', 'P22735', 'P10599', 'Q13867', 'P62979', 'P31944', 'P61626', 'P35030', 'Q96P63', 'P62851', 'A8K2U0', 'P50995']


## 4. Save

Write all four AnnData objects back to `Data/AnnData/`. The objects now carry additional columns computed in this notebook — `completeness_sample_SC/QC` in `obs`, `completeness_feature_{lab}_SC/QC` in `var`, and `completeness_outlier` in `obs` — which are consumed by downstream analysis notebooks for sample filtering and variance modelling.

In [42]:
ANNDATA_OUT = os.path.join(
    ROOT_OUT,
    'AnnData'
)

adata_precursor.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein.h5ad'),
    convert_strings_to_categoricals=False
)

adata_precursor_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor_qc.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein_qc.h5ad'),
    convert_strings_to_categoricals=False
)